# Prepare matched Tractor-Mix / SAIGE pilot inputs

Builds a **shared recommended-model-complete** analysis cohort for four matched
calibration runs (Tractor/SAIGE × limited/full covariates) from
`covariates.source_rebuilt.csv.gz`:

- `analysis_samples.txt` — VCF ∩ phenotype ∩ covariate IDs, VCF order, full-model complete
- `pheno_cov.tsv` — `ID`, both covariate sets, and selected binary phenotypes (workflow input)
- `covariates_limited.tsv` — standalone `ID` + limited covariate matrix
- `covariates_full.tsv` — standalone `ID` + recommended full covariate matrix
- `covariate_columns_limited.txt` — `sex`, `age`, `PC1`–`PC10`
- `covariate_columns_full.txt` — limited + standardized `coverage` + GC dummies
- `technical_covariate_levels.tsv` — GC reference/dummy columns and coverage mean/sd

Here, **full** means the recommended production set. It intentionally excludes
extraction method, platform, methylation caller, and SV counts; use those only
in explicit sensitivity analyses.

FLARE VCF URIs are resolved from the Terra data table `aou_lr_chrom` via the
firecloud API (column `model_chr_anc_vcf`). Workspace:
`allofus-drc-wgs-LR-prodData` / `AoU_DRC_LongReads_PhaseTwo_Storage`.

Phenotypes: `gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scratch/kvg/saige/AoU_Phase2_Phenotype.csv.gz`

Confirmed FLARE ancestries: `eas=0,amr=1,eur=2,afr=3,sas=4` → `num_ancs=5`.

Upload the source-rebuilt covariate table and set `TRACTOR_COVARIATES_GCS` to its URI before running this notebook.

**Terra WDL:** After this notebook, submit `TractorMixPilot.wdl` with Docker
`tractor-mix-pilot:0.4.2` (shared `docker` input). FitNull is R (`fit_null.R`, ~3h/phenotype);
Score is `tractor-mix-score --threads 8` (~30–45 min/phenotype on chr22). The final
cell stages required WDL scripts to `$WORKSPACE_BUCKET/scripts/` — do not use legacy
`fit_null_and_score.R`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Bootstrap scripts/ from $WORKSPACE_BUCKET/scripts/ when not on the VM.
for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError(
            "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
            "Upload scripts/ to gs://WORKSPACE/scripts/."
        )
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        ["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"]
    )
    sys.path.insert(0, str(_dest))

from terra_notebook import init_notebook
SCRIPTS = init_notebook(
    "workspace_paths.py",
    "resolve_flare_uris.py",
    "select_phenotypes.py",
)
from workspace_paths import data_root

ROOT = data_root()
select_py = SCRIPTS / "select_phenotypes.py"
assert select_py.is_file(), select_py
assert "covariates_limited.tsv" in select_py.read_text(), (
    f"{select_py} is outdated (no covariates_limited.tsv)."
)
print("ROOT:", ROOT)
print("scripts:", sorted(p.name for p in SCRIPTS.glob("*")))


In [ ]:
from pathlib import Path
import json
import os

import pandas as pd

from resolve_flare_uris import (
    DEFAULT_ENTITY_TYPE,
    DEFAULT_NAMESPACE,
    DEFAULT_WORKSPACE,
    fetch_chrom_table_firecloud,
    resolve_uris,
)

OUT_DIR = Path("tractor_mix_pilot_inputs")
OUT_DIR.mkdir(exist_ok=True)

# Terra workspace hosting the aou_lr_chrom data table
TERRA_NAMESPACE = os.environ.get("TERRA_NAMESPACE", DEFAULT_NAMESPACE)
TERRA_WORKSPACE = os.environ.get("TERRA_WORKSPACE", DEFAULT_WORKSPACE)
TERRA_ENTITY_TYPE = os.environ.get("TERRA_ENTITY_TYPE", DEFAULT_ENTITY_TYPE)

print(f"Fetching {TERRA_ENTITY_TYPE} from {TERRA_NAMESPACE}/{TERRA_WORKSPACE} ...")
chrom_table = fetch_chrom_table_firecloud(
    TERRA_NAMESPACE, TERRA_WORKSPACE, TERRA_ENTITY_TYPE
)
print(f"Loaded {len(chrom_table)} chromosomes: {sorted(chrom_table)}")

uris = resolve_uris(
    chrom_table,
    scan_chrom="chr22",
    grm_chroms=["chr1", "chr22"],
    uri_column="model_chr_anc_vcf",
)
(OUT_DIR / "flare_uris.json").write_text(json.dumps(uris, indent=2) + "\n")

FLARE_VCF = os.environ.get("TRACTOR_FLARE_VCF", uris["flare_vcf"])
GRM_VCFS = uris["grm_vcfs"]

PHENOTYPE_GCS = os.environ.get(
    "TRACTOR_PHENOTYPE_GCS",
    "gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scratch/kvg/saige/AoU_Phase2_Phenotype.csv.gz",
)
PHENOTYPE_CSV = Path("AoU_Phase2_Phenotype.csv.gz")
# Upload the source-rebuilt covariate table to this URI before running on Terra,
# or set TRACTOR_COVARIATES_GCS to its location. Do not fall back to covariates.v2.
COVARIATES_GCS = os.environ.get(
    "TRACTOR_COVARIATES_GCS",
    f"{os.environ.get('WORKSPACE_BUCKET', '').rstrip('/')}/covariates/covariates.source_rebuilt.csv.gz"
    if os.environ.get("WORKSPACE_BUCKET")
    else "",
)
COVARIATES_CSV = Path("covariates.source_rebuilt.csv.gz")
# ##ANCESTRY=<eas=0,amr=1,eur=2,afr=3,sas=4>
NUM_ANCS = int(os.environ.get("TRACTOR_NUM_ANCS", "5"))
N_PHENOTYPES = 10
MIN_CASES = 100
N_PCS = 10

print("FLARE_VCF (chr22 scan):", FLARE_VCF)
print("GRM_VCFS (chr1+chr22):")
for u in GRM_VCFS:
    print(" ", u)
print("PHENOTYPE_GCS:", PHENOTYPE_GCS)
print("COVARIATES_CSV:", COVARIATES_CSV)
print("NUM_ANCS:", NUM_ANCS)
print("OUT_DIR:", OUT_DIR.resolve())
display(uris)

In [ ]:
def sh(cmd: str) -> None:
    print(cmd)
    rc = get_ipython().system(cmd)
    if rc:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")


ws = os.environ.get("WORKSPACE_BUCKET", "")

# Pull phenotype / source-rebuilt covariates into the notebook cwd if needed.
# The URI is intentionally explicit: silently using the legacy v2 table would
# invalidate the rebuilt-table contract.
if not COVARIATES_CSV.exists() and COVARIATES_GCS:
    sh(f"gsutil cp {COVARIATES_GCS} {COVARIATES_CSV}")

if not PHENOTYPE_CSV.exists():
    sh(f"gsutil cp {PHENOTYPE_GCS} {PHENOTYPE_CSV}")

assert COVARIATES_CSV.exists(), (
    f"Missing {COVARIATES_CSV}. Upload the rebuilt table and set "
    "TRACTOR_COVARIATES_GCS=gs://.../covariates.source_rebuilt.csv.gz, "
    "or place it in the notebook working directory."
)
assert PHENOTYPE_CSV.exists(), f"Missing {PHENOTYPE_CSV}"

# Sample IDs from FLARE VCF (stream; do not copy full VCF locally)
vcf_samples = OUT_DIR / "vcf_samples.txt"
if FLARE_VCF.startswith("gs://"):
    sh(f"gsutil cat {FLARE_VCF} | bcftools query -l > {vcf_samples}")
else:
    sh(f"bcftools query -l {FLARE_VCF} > {vcf_samples}")
print(f"Wrote {vcf_samples} ({sum(1 for _ in open(vcf_samples)):,} samples)")

In [ ]:
select_py = SCRIPTS / "select_phenotypes.py"
assert select_py.exists(), select_py

sh(
    f"python3 {select_py} "
    f"--phenotype-csv {PHENOTYPE_CSV} "
    f"--covariates-csv {COVARIATES_CSV} "
    f"--vcf-samples {vcf_samples} "
    f"--out-dir {OUT_DIR} "
    f"--n-phenotypes {N_PHENOTYPES} "
    f"--min-cases {MIN_CASES} "
    f"--n-pcs {N_PCS}"
)

stats = pd.read_csv(OUT_DIR / "selected_phenotype_stats.tsv", sep="\t")
display(stats)
pheno_cov = pd.read_csv(OUT_DIR / "pheno_cov.tsv", sep="\t", nrows=5)
display(pheno_cov)
limited_matrix = pd.read_csv(OUT_DIR / "covariates_limited.tsv", sep="\t", nrows=5)
full_matrix = pd.read_csv(OUT_DIR / "covariates_full.tsv", sep="\t", nrows=5)
display(limited_matrix)
display(full_matrix)
print("limited covariates:", (OUT_DIR / "covariate_columns_limited.txt").read_text().split())
print("full covariates:", (OUT_DIR / "covariate_columns_full.txt").read_text().split())
print("legacy covariate_columns:", (OUT_DIR / "covariate_columns.txt").read_text().split())
levels = pd.read_csv(OUT_DIR / "technical_covariate_levels.tsv", sep="\t")
display(levels)
n_samples = sum(1 for _ in open(OUT_DIR / "analysis_samples.txt"))
print(f"shared full-complete analysis samples: {n_samples:,}")
print(f"num_ancs (for WDL inputs): {NUM_ANCS}")

In [ ]:
# Upload prep outputs to the workspace bucket for Terra / Cromwell
samples_path = OUT_DIR / "analysis_samples.txt"
n_upload = sum(1 for line in samples_path.read_text().splitlines() if line.strip())
assert n_upload > 1000, (
    f"Refusing upload: {samples_path} has only {n_upload} IDs "
    f"({samples_path.stat().st_size} bytes). Re-run select_phenotypes first."
)

if ws:
    dest = f"{ws}/tractor_mix_pilot/"
    for name in [
        "analysis_samples.txt",
        "pheno_cov.tsv",
        "covariates_limited.tsv",
        "covariates_full.tsv",
        "selected_phenotypes.txt",
        "selected_phenotype_stats.tsv",
        "covariate_columns_limited.txt",
        "covariate_columns_full.txt",
        "covariate_columns.txt",
        "technical_covariate_levels.tsv",
        "vcf_samples.txt",
        "flare_uris.json",
    ]:
        path = OUT_DIR / name
        if path.exists():
            sh(f"gsutil cp {path} {dest}{name}")
    sh(f"gsutil ls -l {dest}analysis_samples.txt")
    sh(f"gsutil cat {dest}analysis_samples.txt | wc -l")
    print("Uploaded to", dest)
    print("Use the same shared cohort for all four matched runs:")
    print(f"  analysis_samples:        {dest}analysis_samples.txt")
    print(f"  pheno_cov:               {dest}pheno_cov.tsv")
    print(f"  selected_phenotypes:     {dest}selected_phenotypes.txt")
    print(f"  covariates_limited:      {dest}covariates_limited.tsv")
    print(f"  covariates_full:         {dest}covariates_full.tsv")
    print(f"  columns (limited):       {dest}covariate_columns_limited.txt")
    print(f"  columns (full):          {dest}covariate_columns_full.txt")
    print(f"  technical levels:        {dest}technical_covariate_levels.tsv")
    print("Tractor-Mix / SAIGE: use pheno_cov.tsv plus the matching covariate_columns file.")
else:
    print("WORKSPACE_BUCKET unset; outputs left in", OUT_DIR.resolve())

In [ ]:
# Stage WDL scripts for TractorMixPilot / SaigePilot (fit_null.R + helpers)
WDL_SCRIPTS = [
    "fit_null.R",
    "fit_saige_null.R",
    "reorder_dosages.py",
    "sparsify_grm.R",
    "make_plink_keep.py",
    "run_saige_step2.R",
    "build_saige_plink_and_grm.sh",
    "compare_calibration.py",
    "plot_tractor_results.py",
    "resolve_flare_uris.py",
    "select_phenotypes.py",
    "terra_notebook.py",
    "workspace_paths.py",
]

if ws:
    script_dest = f"{ws}/scripts/"
    for name in WDL_SCRIPTS:
        local = SCRIPTS / name
        assert local.is_file(), f"missing {local} — sync repo scripts/ first"
        sh(f"gsutil cp {local} {script_dest}{name}")
    sh(f"gsutil ls -l {script_dest}fit_null.R")
    print("Staged WDL scripts to", script_dest)
    print("TractorMixPilot: FitNull -> fit_null.R; Score -> tractor-mix-score --threads 8 (docker 0.4.2)")
    print("Input JSON: TractorMixPilot.fit_null_script =", f"{script_dest}fit_null.R")
else:
    print("WORKSPACE_BUCKET unset; skip WDL script upload")